# 💻 Lecture 5 Activity Notebook: Aggregation & Data Cleaning
## AA640: Data Analytics and Text Mining
### Bryant University | Prof. Gianluca Brero

---

**IMPORTANT:** *Before starting, save a copy to your Drive via `File > "Save a copy in Drive"`*

### 🎯 Estimated Time: 90 Minutes

#### In-Class Schedule

| Time | Clock | Block |
|------|-------|-------|
| 30 min | 6:30pm | Lecture: Aggregation & Grouping |
| 20 min | 7:00pm | **Activity 1** — GroupBy, `.agg()` & Pivot Tables |
| 30 min | 7:20pm | Lecture: Data Cleaning |
| 30 min | 7:50pm | Break + Quiz |
| 20 min | 8:20pm | **Activity 2** — Missing Values, Duplicates & Outliers |
| 30 min | 8:40pm | Open time — questions, finish notebook |

#### After-Class Activities (~30 min)
*Complete these on your own after lecture.*

| Activity | Topic | Time |
|----------|-------|------|
| Activity 3 | GroupBy Interpretation & Pivot Table Challenge | ~15 min |
| Activity 4 | Capstone: Full Cleaning Pipeline | ~15 min |

### ⚙️ How to Use This Notebook
1. **Read** each section carefully
2. **Run** the example cells first to see how things work
3. **Complete** the activities marked with 📝
4. **Check** your answers against the expected output

> 💡 **Tip:** If you get stuck, re-read the example cell right above the activity — the pattern is always there!

## Setup: Import Libraries & Create Sample Data

In [ ]:
import pandas as pd
import numpy as np

# Sample sales dataset (used throughout this notebook)
sales_data = {
    'region':   ['East','East','East','West','West','East','West','East','West','West'],
    'product':  ['A','B','A','A','B','B','A','A','B','A'],
    'salesperson': ['Alice','Bob','Alice','Charlie','Charlie','Bob','Alice','Charlie','Bob','Charlie'],
    'sales':    [100, 150, 180, 90, 120, 200, 60, 130, 80, 110],
    'quantity': [10, 12, 15, 8, 10, 18, 5, 11, 7, 9]
}
df = pd.DataFrame(sales_data)
print("Sales DataFrame:")
print(df)
print(f"\nShape: {df.shape}")

Sales DataFrame:
  region product salesperson  sales  quantity
0   East       A       Alice    100        10
1   East       B         Bob    150        12
2   East       A       Alice    180        15
3   West       A     Charlie     90         8
4   West       B     Charlie    120        10
5   East       B         Bob    200        18
6   West       A       Alice     60         5
7   East       A     Charlie    130        11
8   West       B         Bob     80         7
9   West       A     Charlie    110         9

Shape: (10, 5)


---
# 📌 IN-CLASS ACTIVITIES

Complete these sections during class time. Each activity has demo cells above it — run the demos first, then work on the 📝 activity.

---
## Activity 1: Aggregation & Grouping

### Part 1: GroupBy & Split-Apply-Combine

The `groupby()` method follows the **Split-Apply-Combine** pattern:
1. **SPLIT** — divide data into groups by one or more keys
2. **APPLY** — apply a function (sum, mean, count, etc.) to each group
3. **COMBINE** — merge results into a new DataFrame

### Demo: Basic GroupBy Operations

In [ ]:
# Total sales by region
print("=== Total Sales by Region ===")
print(df.groupby('region')['sales'].sum())

print("\n=== Average Sales by Product ===")
print(df.groupby('product')['sales'].mean())

print("\n=== Group by Multiple Columns ===")
print(df.groupby(['region', 'product'])['sales'].sum())

print("\n=== Multiple Aggregations at Once ===")
print(df.groupby('region')['sales'].agg(['sum', 'mean', 'count']))

=== Total Sales by Region ===
region
East    760
West    460
Name: sales, dtype: int64

=== Average Sales by Product ===
product
A    111.666667
B    137.500000
Name: sales, dtype: float64

=== Group by Multiple Columns ===
region  product
East    A          410
        B          350
West    A          260
        B          200
Name: sales, dtype: int64

=== Multiple Aggregations at Once ===
        sum   mean  count
region                   
East    760  152.0      5
West    460   92.0      5


### 📝 ACTIVITY 1a: GroupBy Practice

Using the `df` DataFrame created above, answer these questions using `groupby()`.

Think of it as: *"For each unique value in column X, calculate Y."*

In [ ]:
# ACTIVITY 1a: GroupBy Practice

# Task A: Find the total quantity sold by each salesperson
qty_by_person = df.groupby('salesperson')['quantity'].sum() # Hint: groupby('salesperson')['quantity']...
print("Total quantity by salesperson:")
print(qty_by_person)
# Expected: Alice=30, Bob=37, Charlie=28

print()

# Task B: Find the average sales per region AND per product (group by both)
avg_region_product = df.groupby(['region', 'product'])['sales'].mean()  # Hint: groupby(['region', 'product'])['sales']...
print("Average sales by region & product:")
print(avg_region_product)

print()

# Task C: Count how many transactions each salesperson made per region
count_person_region = df.groupby(['salesperson', 'region'])['sales'].count() # Hint: groupby(['salesperson', 'region'])['sales']...
print("Transaction count by salesperson & region:")
print(count_person_region)

Total quantity by salesperson:
salesperson
Alice      30
Bob        37
Charlie    38
Name: quantity, dtype: int64

Average sales by region & product:
region  product
East    A          136.666667
        B          175.000000
West    A           86.666667
        B          100.000000
Name: sales, dtype: float64

Transaction count by salesperson & region:
salesperson  region
Alice        East      2
             West      1
Bob          East      2
             West      1
Charlie      East      1
             West      3
Name: sales, dtype: int64


### Part 2: `.agg()` & Pivot Tables

The `.agg()` method lets you apply **multiple** or **custom** aggregation functions at once.
Pivot tables reshape grouped data into a spreadsheet-style layout.

### Demo: Named Aggregations & Custom Functions

In [ ]:
# Named aggregations — clean, readable column names!
result = df.groupby('region').agg(
    total_sales=('sales', 'sum'),
    avg_sales=('sales', 'mean'),
    num_orders=('sales', 'count'),
    avg_qty=('quantity', 'mean')
)
print("=== Named Aggregations ===")
print(result)

print("\n=== Custom Function: Range of Sales (max - min) ===")
sales_range = df.groupby('region')['sales'].agg(lambda x: x.max() - x.min())
print(sales_range)

print("\n=== Custom Function: Percentage of Total Sales ===")
def pct_of_total(x):
    return x.sum() / df['sales'].sum() * 100

print(df.groupby('region')['sales'].agg(pct_of_total))

=== Named Aggregations ===
        total_sales  avg_sales  num_orders  avg_qty
region                                             
East            760      152.0           5     13.2
West            460       92.0           5      7.8

=== Custom Function: Range of Sales (max - min) ===
region
East    100
West     60
Name: sales, dtype: int64

=== Custom Function: Percentage of Total Sales ===
region
East    62.295082
West    37.704918
Name: sales, dtype: float64


### Demo: Pivot Tables

In [ ]:
# Basic pivot table
print("=== Pivot Table: Total Sales by Region & Product ===")
pivot = pd.pivot_table(df, values='sales', index='region',
                       columns='product', aggfunc='sum')
print(pivot)

print("\n=== With Totals (margins=True) ===")
pivot_totals = pd.pivot_table(df, values='sales', index='region',
                              columns='product', aggfunc='sum',
                              margins=True, margins_name='Total')
print(pivot_totals)

=== Pivot Table: Total Sales by Region & Product ===
product    A    B
region           
East     410  350
West     260  200

=== With Totals (margins=True) ===
product    A    B  Total
region                  
East     410  350    760
West     260  200    460
Total    670  550   1220


### 📝 ACTIVITY 1b: Custom Aggregations

Create a summary report using `.agg()` with both built-in and custom functions.

In [ ]:
# ACTIVITY 1b: Custom Aggregations

# Task A: Create a named aggregation that shows, for each salesperson:
#   - total_sales (sum of sales)
#   - best_sale (max of sales)
#   - num_transactions (count of sales)
salesperson_summary = df.groupby('salesperson').agg(
    total_sales=('sales', 'sum'),
    best_sale=('sales', 'max'),
    num_transactions=('sales', 'count')
)
print("Salesperson Summary:")
print(salesperson_summary)

print()

# Task B: Write a custom function that calculates what percentage
# of total sales each region accounts for: (group sum / total) * 100
def pct_of_total(x):
    return x.sum() / df['sales'].sum() * 100

pct_by_region = df.groupby('region')['sales'].agg(pct_of_total)
print("Percentage of Total Sales by Region:")
print(pct_by_region) # Return x.sum() / df['sales'].sum() * 100


Salesperson Summary:
             total_sales  best_sale  num_transactions
salesperson                                          
Alice                340        180                 3
Bob                  430        200                 3
Charlie              450        130                 4

Percentage of Total Sales by Region:
region
East    62.295082
West    37.704918
Name: sales, dtype: float64


---
## Activity 2: Data Cleaning

### Part 3: Missing Values & Duplicates

Real-world data is messy! This section covers detecting and handling **missing values** and **duplicates**.

### Demo: Working with Missing Values

In [ ]:
# Create a messy DataFrame with missing values
messy_data = {
    'name':    ['Alice', 'Bob', None, 'Diana', 'Eve', 'Frank'],
    'age':     [25, np.nan, 35, 28, np.nan, 42],
    'dept':    ['Sales', 'Engineering', 'Sales', 'N/A', 'Engineering', ''],
    'salary':  [50000, 65000, 70000, np.nan, 58000, 72000]
}
messy = pd.DataFrame(messy_data)
print("Messy DataFrame:")
print(messy)

print("\n=== Missing Value Counts ===")
print(messy.isnull().sum())

print("\n=== Percentage Missing ===")
print((messy.isnull().mean() * 100).round(1))

# IMPORTANT: Pandas does NOT recognize 'N/A', '', 'NULL' as missing!
messy = messy.replace(['N/A', '', 'NULL', '-'], np.nan)
print("\n=== After replacing placeholders ===")
print(messy.isnull().sum())

Messy DataFrame:
    name   age         dept   salary
0  Alice  25.0        Sales  50000.0
1    Bob   NaN  Engineering  65000.0
2   None  35.0        Sales  70000.0
3  Diana  28.0          N/A      NaN
4    Eve   NaN  Engineering  58000.0
5  Frank  42.0               72000.0

=== Missing Value Counts ===
name      1
age       2
dept      0
salary    1
dtype: int64

=== Percentage Missing ===
name      16.7
age       33.3
dept       0.0
salary    16.7
dtype: float64

=== After replacing placeholders ===
name      1
age       2
dept      2
salary    1
dtype: int64


### Demo: Filling vs Dropping

In [ ]:
# Strategy 1: Drop rows with any missing
print(f"Rows before dropna: {len(messy)}")
dropped = messy.dropna()
print(f"Rows after dropna:  {len(dropped)}")
print(f"Lost {len(messy) - len(dropped)} rows!\n")

# Strategy 2: Fill with mean/median
filled = messy.copy()
filled['age'] = filled['age'].fillna(filled['age'].median())
filled['salary'] = filled['salary'].fillna(filled['salary'].median())
filled['name'] = filled['name'].fillna('Unknown')
filled['dept'] = filled['dept'].fillna('Unknown')
print("After filling:")
print(filled)
print(f"\nMissing values remaining: {filled.isnull().sum().sum()}")

Rows before dropna: 6
Rows after dropna:  1
Lost 5 rows!

After filling:
      name   age         dept   salary
0    Alice  25.0        Sales  50000.0
1      Bob  31.5  Engineering  65000.0
2  Unknown  35.0        Sales  70000.0
3    Diana  28.0      Unknown  65000.0
4      Eve  31.5  Engineering  58000.0
5    Frank  42.0      Unknown  72000.0

Missing values remaining: 0


### 📝 ACTIVITY 2a: Missing Value Detective

Explore and handle missing values in a new dataset.

In [ ]:
# ACTIVITY 2a: Missing Value Detective

# Here's a dataset of student records
students = pd.DataFrame({
    'student_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'name':       ['Ana', 'Ben', 'Cara', None, 'Eli', 'Fay', 'Gus', 'Hana'],
    'major':      ['CS', 'Math', 'CS', 'Stats', '', 'Math', 'CS', 'N/A'],
    'gpa':        [3.8, np.nan, 3.5, 3.2, 3.9, np.nan, 3.1, 3.6],
    'credits':    [90, 60, 120, 45, 85, 30, np.nan, 100]
})

print("Raw data:")
print(students)

# Task A: Replace placeholder strings ('', 'N/A') with NaN
students = students.replace(['', 'N/A'], np.nan)  # Hint: students.replace(...)

# Task B: Print the count and percentage of missing values per column
print("\nMissing counts:")
print(students.isnull().sum())  # Hint: students.isnull().sum()
print("\nMissing percentages:")
print((students.isnull().mean() * 100).round(1))  # Hint: (students.isnull().mean() * 100).round(1)

# Task C: Fill 'gpa' with the median GPA, 'credits' with the mean credits,
#         'name' with 'Unknown', 'major' with 'Undeclared'
students['gpa'] = students['gpa'].fillna(students['gpa'].median())
students['credits'] = students['credits'].fillna(students['credits'].mean())
students['name'] = students['name'].fillna('Unknown')
students['major'] = students['major'].fillna('Undeclared')

# Task D: Verify no missing values remain
print("\nFinal missing count:")
print(students.isnull().sum())
print(f"\nTotal missing: {students.isnull().sum().sum()}")
# Expected: 0 missing values

Raw data:
   student_id  name  major  gpa  credits
0         101   Ana     CS  3.8     90.0
1         102   Ben   Math  NaN     60.0
2         103  Cara     CS  3.5    120.0
3         104  None  Stats  3.2     45.0
4         105   Eli         3.9     85.0
5         106   Fay   Math  NaN     30.0
6         107   Gus     CS  3.1      NaN
7         108  Hana    N/A  3.6    100.0

Missing counts:
student_id    0
name          1
major         2
gpa           2
credits       1
dtype: int64

Missing percentages:
student_id     0.0
name          12.5
major         25.0
gpa           25.0
credits       12.5
dtype: float64

Final missing count:
student_id    0
name          0
major         0
gpa           0
credits       0
dtype: int64

Total missing: 0


### Demo: Detecting and Removing Duplicates

In [ ]:
# Create data with duplicates
dup_data = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob', 'Alice'],
    'email': ['a@x.com', 'b@x.com', 'a@x.com', 'c@x.com', 'b@x.com', 'a@y.com'],
    'score': [85, 90, 85, 78, 90, 92]
})
print("Data with duplicates:")
print(dup_data)

print(f"\nDuplicate rows: {dup_data.duplicated().sum()}")
print(f"Duplicates by name+email: {dup_data.duplicated(subset=['name','email']).sum()}")

# By default, duplicated() marks only the SECOND copy as a duplicate.
# Use keep=False to mark ALL copies (original + duplicates) so you can
# see them side by side and decide which one to keep:
print("\nAll copies (originals + duplicates):")
print(dup_data[dup_data.duplicated(subset=['name','email'], keep=False)])

# Remove duplicates
clean = dup_data.drop_duplicates(subset=['name','email'], keep='first')
print(f"\nAfter dedup: {len(dup_data)} -> {len(clean)} rows")
print(clean)

Data with duplicates:
      name    email  score
0    Alice  a@x.com     85
1      Bob  b@x.com     90
2    Alice  a@x.com     85
3  Charlie  c@x.com     78
4      Bob  b@x.com     90
5    Alice  a@y.com     92

Duplicate rows: 2
Duplicates by name+email: 2

All copies (originals + duplicates):
    name    email  score
0  Alice  a@x.com     85
1    Bob  b@x.com     90
2  Alice  a@x.com     85
4    Bob  b@x.com     90

After dedup: 6 -> 4 rows
      name    email  score
0    Alice  a@x.com     85
1      Bob  b@x.com     90
3  Charlie  c@x.com     78
5    Alice  a@y.com     92


### Part 4: Outliers

**Outliers** are values that are surprisingly far from the rest — they can be typos, data entry errors, or genuinely extreme values. Either way, they can skew your analysis.

### Demo: Spotting and Handling Outliers

Start with `.describe()` — look at the **min** and **max**. Do they make sense?

If something looks off, you can use a built-in rule to automatically flag values that are "too far" from the rest. The rule works like this:
1. Sort all the values and find where the **bottom 25%** and **top 25%** fall
2. Measure the **spread** between those two points
3. Anything more than **1.5× that spread** beyond those points is flagged

In [ ]:
# Employee salary data — two values look suspicious!
salaries = pd.DataFrame({
    'employee': ['A','B','C','D','E','F','G','H','I','J'],
    'salary':   [45000, 52000, 48000, 55000, 300000, 50000, 47000, 51000, -5000, 53000]
})

# Step 1: Start with .describe() — do min and max make sense?
print("=== Quick Look ===")
print(salaries['salary'].describe())
print("\n$300,000 and -$5,000 look suspicious!")

# Step 2: Use the rule to set boundaries
bottom_25 = salaries['salary'].quantile(0.25)   # Where the bottom 25% ends
top_25    = salaries['salary'].quantile(0.75)    # Where the top 25% starts
spread    = top_25 - bottom_25                   # How spread out the middle is

lower = bottom_25 - 1.5 * spread   # Anything below this is flagged
upper = top_25    + 1.5 * spread   # Anything above this is flagged
print(f"\nAcceptable range: [${lower:,.0f}, ${upper:,.0f}]")

# Step 3: Find the outliers
outliers = salaries[(salaries['salary'] < lower) | (salaries['salary'] > upper)]
print(f"\nOutliers found ({len(outliers)}):")
print(outliers)

# Step 4: Decide what to do — cap them at the boundaries
salaries['salary_capped'] = salaries['salary'].clip(lower=lower, upper=upper)
print("\nAfter capping extreme values:")
print(salaries[['employee', 'salary', 'salary_capped']])

=== Quick Look ===
count        10.000000
mean      69600.000000
std       82838.799283
min       -5000.000000
25%       47250.000000
50%       50500.000000
75%       52750.000000
max      300000.000000
Name: salary, dtype: float64

$300,000 and -$5,000 look suspicious!

Acceptable range: [$39,000, $61,000]

Outliers found (2):
  employee  salary
4        E  300000
8        I   -5000

After capping extreme values:
  employee  salary  salary_capped
0        A   45000          45000
1        B   52000          52000
2        C   48000          48000
3        D   55000          55000
4        E  300000          61000
5        F   50000          50000
6        G   47000          47000
7        H   51000          51000
8        I   -5000          39000
9        J   53000          53000


### 📝 ACTIVITY 2b: Outlier Detective

Spot suspicious values and decide what to do with them.

In [ ]:
# ACTIVITY 2b: Outlier Detective

# Here's a dataset of employee reviews
reviews = pd.DataFrame({
    'employee': ['Ana', 'Ben', 'Cara', 'Dan', 'Eli', 'Fay', 'Gus', 'Hana'],
    'rating':   [4.2, 3.8, 4.5, -1.0, 3.9, 4.1, 99.0, 3.7],
    'salary':   [52000, 48000, 61000, 55000, 950000, 50000, 47000, 53000]
})

print("Raw data:")
print(reviews)

# Task A: Use .describe() to spot suspicious values in 'rating' and 'salary'
print("\nRating stats:")
print(reviews['rating'].describe())  # Hint: reviews['rating'].describe()
print("\nSalary stats:")
print(reviews['salary'].describe())  # Hint: reviews['salary'].describe()
# What looks off? Dan has -1.0 rating, Gus has 99.0, Eli has $950k

# Task B: Replace impossible ratings (below 0 or above 5) with NaN
reviews.loc[(reviews['rating'] < 0) | (reviews['rating'] > 5), 'rating'] = np.nan  # Hint: reviews.loc[condition, 'rating'] = np.nan

# Task C: Use the outlier rule to find suspicious salaries
bottom_25 = reviews['salary'].quantile(0.25)
top_25    = reviews['salary'].quantile(0.75)
spread    = top_25 - bottom_25
lower = bottom_25 - 1.5 * spread
upper = top_25    + 1.5 * spread

print(f"\nSalary acceptable range: [${lower:,.0f}, ${upper:,.0f}]")
salary_outliers = reviews[(reviews['salary'] < lower) | (reviews['salary'] > upper)]  # Hint: reviews[(reviews['salary'] < lower) | (reviews['salary'] > upper)]
print(f"Salary outliers:\n{salary_outliers}")

# Task D: Cap the salary outliers at the boundaries
reviews['salary'] = reviews['salary'].clip(lower=lower, upper=upper)  # Hint: reviews['salary'].clip(lower=lower, upper=upper)

# Task E: Fill the NaN ratings with the median rating
reviews['rating'] = reviews['rating'].fillna(reviews['rating'].median())  # Hint: reviews['rating'].fillna(...)

print("\nCleaned data:")
print(reviews)

Raw data:
  employee  rating  salary
0      Ana     4.2   52000
1      Ben     3.8   48000
2     Cara     4.5   61000
3      Dan    -1.0   55000
4      Eli     3.9  950000
5      Fay     4.1   50000
6      Gus    99.0   47000
7     Hana     3.7   53000

Rating stats:
count     8.000000
mean     15.275000
std      33.876741
min      -1.000000
25%       3.775000
50%       4.000000
75%       4.275000
max      99.000000
Name: rating, dtype: float64

Salary stats:
count         8.000000
mean     164500.000000
std      317420.406221
min       47000.000000
25%       49500.000000
50%       52500.000000
75%       56500.000000
max      950000.000000
Name: salary, dtype: float64

Salary acceptable range: [$39,000, $67,000]
Salary outliers:
  employee  rating  salary
4      Eli     3.9  950000

Cleaned data:
  employee  rating  salary
0      Ana     4.2   52000
1      Ben     3.8   48000
2     Cara     4.5   61000
3      Dan     4.0   55000
4      Eli     3.9   67000
5      Fay     4.1   50000
6  

---
# 🏠 AFTER-CLASS ACTIVITIES

Complete these on your own after lecture (~30 min total).

---
## Activity 3: GroupBy Interpretation & Pivot Table Challenge (~15 min)

### 📝 ACTIVITY 3a: Interpret GroupBy Results

Run the cell below and answer the questions.

In [ ]:
import pandas as pd
import numpy as np

# Sample sales dataset (used throughout this notebook)
sales_data = {
    'region':   ['East','East','East','West','West','East','West','East','West','West'],
    'product':  ['A','B','A','A','B','B','A','A','B','A'],
    'salesperson': ['Alice','Bob','Alice','Charlie','Charlie','Bob','Alice','Charlie','Bob','Charlie'],
    'sales':    [100, 150, 180, 90, 120, 200, 60, 130, 80, 110],
    'quantity': [10, 12, 15, 8, 10, 18, 5, 11, 7, 9]
}
df = pd.DataFrame(sales_data)

# ACTIVITY 3a: Interpret these results

summary = df.groupby('region')['sales'].agg(['sum', 'mean', 'min', 'max', 'count'])
print(summary)

# QUESTION 1: Which region has higher TOTAL sales?
# Your answer: East (760 vs 460)

# QUESTION 2: Which region has a higher AVERAGE sale amount?
# Your answer: East (152.0 vs 92.0)

# QUESTION 3: Why might total and average tell different stories?
# Your answer: Total sales indicate overall volume, while average sales indicate the value per transaction. If regions have different numbers of transactions (counts), the region with a higher total might not have a higher average, or vice-versa. For example, a region could have high total sales from many small transactions, while another has lower total sales from fewer but larger transactions, leading to a higher average.

        sum   mean  min  max  count
region                             
East    760  152.0  100  200      5
West    460   92.0   60  120      5


### 📝 ACTIVITY 3b: Pivot Table Challenge

Create pivot tables to answer business questions.

In [ ]:
# ACTIVITY 3b: Pivot Table Challenge

# Task A: Create a pivot table showing AVERAGE sales by region (rows)
# and product (columns), with totals
pivot_avg = pd.pivot_table(df, values='sales', index='region',
                       columns='product', aggfunc='mean', margins=True, margins_name='Total')
print("Average Sales Pivot:")
print(pivot_avg)

print()

# Task B: Create a pivot table showing total QUANTITY
# by salesperson (rows) and region (columns)
pivot_qty = pd.pivot_table(df, values='quantity', index='salesperson',
                       columns='region', aggfunc='sum', margins=True, margins_name='Total')
print("Quantity by Salesperson & Region:")
print(pivot_qty)

print()

# Task C: Which combination of salesperson + region has the highest total quantity?
# Your answer: (You will need to analyze the 'pivot_qty' output after execution to answer this)

Average Sales Pivot:
product           A      B  Total
region                           
East     136.666667  175.0  152.0
West      86.666667  100.0   92.0
Total    111.666667  137.5  122.0

Quantity by Salesperson & Region:
region       East  West  Total
salesperson                   
Alice          25     5     30
Bob            30     7     37
Charlie        11    27     38
Total          66    39    105



---
## Activity 4: Capstone — Full Cleaning Pipeline (~15 min)

### Scenario: Employee Database Cleanup

You are a data analyst at a company. HR has given you a messy employee database.
Your task: **find and fix missing values, remove duplicates, and flag outliers.**

This exercise combines every technique from today's lecture and mirrors exactly
what you'll do for **Project #1**.

**Document every decision!**

In [ ]:
# === THE MESSY DATASET ===

employees = pd.DataFrame({
    'name':       ['Alice', 'Bob', 'Charlie', 'Alice', 'David',
                   None, 'Eve', 'Frank', 'Grace', 'Bob'],
    'department': ['Sales', 'Engineering', 'Sales', 'Sales', 'N/A',
                   'Engineering', 'Marketing', 'Engineering', '', 'Engineering'],
    'age':        [28, 35, -3, 28, 45, 200, 31, np.nan, 29, 35],
    'salary':     [65000, 82000, 55000, 65000, 90000,
                   78000, 71000, np.nan, 68000, 82000]
})

print("=" * 60)
print("RAW EMPLOYEE DATA")
print("=" * 60)
print(employees)
print(f"\nShape: {employees.shape}")
print(f"\nData types:\n{employees.dtypes}")

RAW EMPLOYEE DATA
      name   department    age   salary
0    Alice        Sales   28.0  65000.0
1      Bob  Engineering   35.0  82000.0
2  Charlie        Sales   -3.0  55000.0
3    Alice        Sales   28.0  65000.0
4    David          N/A   45.0  90000.0
5     None  Engineering  200.0  78000.0
6      Eve    Marketing   31.0  71000.0
7    Frank  Engineering    NaN      NaN
8    Grace                29.0  68000.0
9      Bob  Engineering   35.0  82000.0

Shape: (10, 4)

Data types:
name           object
department     object
age           float64
salary        float64
dtype: object


### 📝 Step 1: Initial Exploration

Before cleaning, understand what you're working with.

In [ ]:
# STEP 1: Explore the data

# Task: Print the following diagnostics:
# - Number of missing values per column
# - Number of duplicate rows
# - Basic statistics for numeric columns
# - Unique values in 'department'

print("=== Missing Values ===")
print(employees.isnull().sum())

print("\n=== Duplicate Rows ===")
print(employees.duplicated().sum())

print("\n=== Basic Stats ===")
print(employees.describe())

print("\n=== Unique Departments ===")
print(employees['department'].unique())

# What issues did you find? List them here:
# 1. Missing values in 'name', 'age', 'salary' and 'department'.
# 2. Duplicate rows exist.
# 3. 'age' column has impossible values (-3, 200) and 'salary' has a missing value.
# 4. 'department' has inconsistent representations for missing values ('N/A', '') and also actual missing (None).
# 5. Some numerical columns are floats, which might be an issue if they should be integers.

=== Missing Values ===
name          1
department    0
age           1
salary        1
dtype: int64

=== Duplicate Rows ===
2

=== Basic Stats ===
              age        salary
count    9.000000      9.000000
mean    47.555556  72888.888889
std     58.630434  10936.686478
min     -3.000000  55000.000000
25%     28.000000  65000.000000
50%     31.000000  71000.000000
75%     35.000000  82000.000000
max    200.000000  90000.000000

=== Unique Departments ===
['Sales' 'Engineering' 'N/A' 'Marketing' '']


### 📝 Step 2: Handle Missing Values

In [ ]:
# STEP 2: Handle missing values
df_clean = employees.copy()  # Always work on a copy!

# Task A: Replace placeholder strings ('N/A', '') with NaN
df_clean['department'] = df_clean['department'].replace(['N/A', '', 'NULL'], np.nan)

# Task B: Fill missing 'name' with 'Unknown', missing 'department' with 'Unknown'
df_clean['name'] = df_clean['name'].fillna('Unknown')
df_clean['department'] = df_clean['department'].fillna('Unknown')

# Task C: How many missing values remain?
print("Missing values after Step 2:")
print(df_clean.isnull().sum())

Missing values after Step 2:
name          0
department    0
age           1
salary        1
dtype: int64


### 📝 Step 3: Handle Duplicates

In [ ]:
# STEP 3: Remove duplicates

print(f"Rows before dedup: {len(df_clean)}")

# Task: Remove duplicates based on 'name', keep first
# DOCUMENT: Why did you choose this column? Why keep first?
df_clean = df_clean.drop_duplicates(subset=['name'], keep='first')

print(f"Rows after dedup:  {len(df_clean)}")
# Cleaning log:
# I chose 'name' as the subset for duplicate removal because in an employee database, each employee should ideally have a unique name or at least a unique primary identifier. If multiple entries exist for the same name, it's likely a duplicate record for the same person. Keeping the 'first' record assumes that the first entry is the most complete or most accurate, or simply establishes a consistent rule for deduplication. This helps in retaining a single, representative record for each employee.

Rows before dedup: 10
Rows after dedup:  8


### 📝 Step 4: Handle Invalid Values & Outliers

In [ ]:
# STEP 4: Fix invalid values and handle outliers

# Task A: Fix ages — replace impossible values (< 18 or > 100) with NaN
# DOCUMENT: Why are these impossible? (age -3 = typo, age 200 = impossible)
df_clean.loc[(df_clean['age'] < 18) | (df_clean['age'] > 100), 'age'] = np.nan  # Hint: condition for invalid ages

# Task B: Fill missing ages (including the ones you just set to NaN) with the median
median_age = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(median_age)
print(f"Ages after cleaning: {df_clean['age'].tolist()}")

# Task C: Fill missing salary with the median salary
df_clean['salary'] = df_clean['salary'].fillna(df_clean['salary'].median())  # Hint: .fillna(df_clean['salary'].median())
print(f"\nSalary missing after fill: {df_clean['salary'].isna().sum()}")

# Task D: Use .describe() to check if salary has outliers
print("\nSalary stats:")
print(df_clean['salary'].describe())
# Do min and max look reasonable?

print("\nAfter all cleaning:")
print(df_clean)
print(f"\nRemaining missing: {df_clean.isnull().sum().sum()}")

Ages after cleaning: [28.0, 35.0, 31.0, 45.0, 31.0, 31.0, 31.0, 29.0]

Salary missing after fill: 0

Salary stats:
count        8.00000
mean     72500.00000
std      10783.58541
min      55000.00000
25%      67250.00000
50%      71000.00000
75%      79000.00000
max      90000.00000
Name: salary, dtype: float64

After all cleaning:
      name   department   age   salary
0    Alice        Sales  28.0  65000.0
1      Bob  Engineering  35.0  82000.0
2  Charlie        Sales  31.0  55000.0
4    David      Unknown  45.0  90000.0
5  Unknown  Engineering  31.0  78000.0
6      Eve    Marketing  31.0  71000.0
7    Frank  Engineering  31.0  71000.0
8    Grace      Unknown  29.0  68000.0

Remaining missing: 0


### 📝 Step 5: Analyze the Clean Data

In [ ]:
# STEP 5: Now analyze!

# Task A: Average salary by department
print("=== Average Salary by Department ===")
print(df_clean.groupby('department')['salary'].mean())

print()

# Task B: Employee count by department
print("=== Employee Count by Department ===")
print(df_clean.groupby('department')['name'].count())  # Hint: groupby('department')['name'].count()

print()

# Task C: Who is the highest-paid employee?
print("=== Highest Paid ===")
highest_paid_employee = df_clean.loc[df_clean['salary'].idxmax()]
print(highest_paid_employee)  # Hint: df_clean.loc[df_clean['salary'].idxmax()]

print()

# BONUS: Print a final cleaning summary
print("=" * 50)
print("CLEANING SUMMARY")
print("=" * 50)
print(f"Original rows:  10")
print(f"Final rows:     {len(df_clean)}")
print(f"Rows removed:   {10 - len(df_clean)}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

=== Average Salary by Department ===
department
Engineering    77000.0
Marketing      71000.0
Sales          60000.0
Unknown        79000.0
Name: salary, dtype: float64

=== Employee Count by Department ===
department
Engineering    3
Marketing      1
Sales          2
Unknown        2
Name: name, dtype: int64

=== Highest Paid ===
name            David
department    Unknown
age              45.0
salary        90000.0
Name: 4, dtype: object

CLEANING SUMMARY
Original rows:  10
Final rows:     8
Rows removed:   2
Missing values: 0


---
# Preview: String Cleaning & Type Conversion *(Next Lecture)*

The demo below shows what we'll cover next time — **you do not need to complete this now.** Just take a quick look so you know what's coming!

In [ ]:
# PREVIEW — String Cleaning & Type Conversion (next lecture)

# Messy string data
contacts = pd.DataFrame({
    'name':   ['  alice SMITH ', 'BOB jones', '  Charlie Brown', 'alice smith  '],
    'phone':  ['555-123-4567', '555.987.6543', '(555) 111-2222', '555-123-4567'],
    'salary': ['$52,000', '$65,000', '$48,500', '$52,000'],
    'date':   ['01/15/2024', '02/28/2024', '03/10/2024', '01/15/2024']
})
print("Before cleaning:")
print(contacts)
print(f"Types: {dict(contacts.dtypes)}")

# Clean names: strip whitespace, title case
contacts['name'] = contacts['name'].str.strip().str.title()

# Clean phone: remove non-digits
contacts['phone'] = contacts['phone'].str.replace(r'[^\d]', '', regex=True)

# Convert salary: remove $ and commas, convert to float
contacts['salary'] = pd.to_numeric(
    contacts['salary'].str.replace(r'[$,]', '', regex=True),
    errors='coerce'
)

# Convert dates
contacts['date'] = pd.to_datetime(contacts['date'], format='%m/%d/%Y')

print("\nAfter cleaning:")
print(contacts)
print(f"\nTypes: {dict(contacts.dtypes)}")

Before cleaning:
              name           phone   salary        date
0     alice SMITH     555-123-4567  $52,000  01/15/2024
1        BOB jones    555.987.6543  $65,000  02/28/2024
2    Charlie Brown  (555) 111-2222  $48,500  03/10/2024
3    alice smith      555-123-4567  $52,000  01/15/2024
Types: {'name': dtype('O'), 'phone': dtype('O'), 'salary': dtype('O'), 'date': dtype('O')}

After cleaning:
            name       phone  salary       date
0    Alice Smith  5551234567   52000 2024-01-15
1      Bob Jones  5559876543   65000 2024-02-28
2  Charlie Brown  5551112222   48500 2024-03-10
3    Alice Smith  5551234567   52000 2024-01-15

Types: {'name': dtype('O'), 'phone': dtype('O'), 'salary': dtype('int64'), 'date': dtype('<M8[ns]')}


---
# ✅ Summary: What You Practiced Today

| Concept | Key Takeaway |
|---------|--------------|
| **`groupby()`** | Split-Apply-Combine for aggregation |
| **`.agg()`** | Multiple/custom aggregation functions |
| **`pivot_table()`** | 2D summary tables with margins |
| **Missing Values** | `isnull()`, `dropna()`, `fillna()` — document your choices! |
| **Duplicates** | `duplicated()`, `drop_duplicates()` — check subset columns |
| **Outliers** | Use `describe()` to spot suspicious values, then remove, cap, or keep |

*String cleaning and type conversion will be covered in the next lecture.*

### 📣 Before Next Week
- **Submit** this notebook via Canvas before the start of next week's lecture
- **Review**: Matplotlib, Seaborn basics
- **Reading**: McKinney Ch. 7 (Data Cleaning) & Ch. 10 (Aggregation)

### 📢 Reminders
- **Office Hours:** Mon 5:15–6:15pm, Wed 12:00–4:00pm
- **PROJECT #1 DUE: March 23rd** — Start cleaning your chosen dataset NOW!

**Great work today!**